## M2 Minimal REAL Loop
Runs a single M2 live session against inference internals and writes standardized artifacts.

In [ ]:
import sys
from pathlib import Path


def find_phase5_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "plan.md").exists() and (candidate / "README.md").exists():
            return candidate
        nested = candidate / "Phase 2" / "Phase 5"
        if (nested / "plan.md").exists():
            return nested
    raise FileNotFoundError("Could not locate Phase 5 root from current working directory.")


PHASE5_ROOT = find_phase5_root(Path.cwd())
PHASE2_ROOT = PHASE5_ROOT.parent
PHASE4_ROOT = PHASE2_ROOT / "Phase 4"

for p in [PHASE5_ROOT, PHASE4_ROOT]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Phase 5 root:", PHASE5_ROOT)
print("Phase 4 root:", PHASE4_ROOT)

In [ ]:
from real_inference.m2 import M2RunConfig, run_m2_minimal_session

# Keep this config block small and stable for low-friction runs.
config = M2RunConfig(
    model_key="tinyllama_1_1b",
    fallback_model_keys=("qwen3_0_6b",),
    prompt_id="cp_001",
    run_mode="smoke",  # smoke | full
    seed=42,
    initial_temperature=0.8,
    interventions_enabled=True,
    output_tag_suffix="v1",
)

result = run_m2_minimal_session(
    phase5_root=PHASE5_ROOT,
    config=config,
)

In [ ]:
print("M2 summary:")
print(result["summary"])
print("Results dir:", result["results_dir"])

if result["summary"].get("m2_success_gco_variation"):
    print("M2 success criterion met: GCO varied across cycles")
else:
    print("M2 not yet at success criterion: GCO did not vary")

In [ ]:
# Optional control run (passive mode) for quick intervention-vs-no-intervention comparison.
passive_config = M2RunConfig(
    model_key=config.model_key,
    fallback_model_keys=config.fallback_model_keys,
    prompt_id=config.prompt_id,
    run_mode=config.run_mode,
    seed=config.seed,
    initial_temperature=config.initial_temperature,
    interventions_enabled=False,
    output_tag_suffix="control",
)

passive_result = run_m2_minimal_session(
    phase5_root=PHASE5_ROOT,
    config=passive_config,
)

print("\nPassive summary:")
print(passive_result["summary"])
print("Passive results dir:", passive_result["results_dir"])
